# Raw GWAS Catalog and GWAS Harvest

This notebook executes the real public manifest, gzip reader, Parquet writer,
resume, and verification APIs using tiny synthetic local files. All studies and
measurements are illustrative. The sources are distinct: Catalog uses exact EBI
harmonised manifest paths; Harvest uses its own observed SNP/CHR/BP/A1/A2 schema
and keeps the full study-directory identity as well as the GCST accession.

Raw readers default to **genomewide with no significance filter**. A lead-only
Open Targets table is a different dataset; see [Open Targets](02_opentargets.ipynb).


In [1]:
import gzip
import json
import tempfile
from pathlib import Path

import polars as pl

work = Path(tempfile.mkdtemp(prefix="biodb-tutorial-"))

In [2]:
from biodb import gwas_catalog, gwas_harvest

header = "SNP\tCHR\tBP\tA1\tA2\tFRQ\tBETA\tSE\tZ\tP\tN\n"
row = "rs1\tX\t100\tA\tG\t0.0545003\t-3.07292\t\t-1.35001010447145\t0\t40943\n"
weak = row.replace("rs1", "rs2").replace("\t0\t40943", "\t0.8\t40943")
directory = work / "GCST000001" / "harmonised"
directory.mkdir(parents=True)
source = directory / "GCST000001.h.tsv.gz"
# Concatenated gzip members are valid and must all be consumed to EOF.
source.write_bytes(gzip.compress((header + row).encode()) + gzip.compress(weak.encode()))
alias = directory / "alias.h.tsv.gz"
alias.write_bytes(gzip.compress((header + weak).encode()))
manifest = work / "harmonised_list.txt"
manifest.write_text(source.as_uri() + "\n" + alias.as_uri() + "\n" + source.as_uri() + ".tbi\n")
files = gwas_catalog.list_datasets(manifest)
assert files.height == 2 and files["file_id"].n_unique() == 2
assert files["study_id"].to_list() == ["GCST000001", "GCST000001"]
print(files.select("source", "study_id"))

shape: (2, 2)
┌──────────────┬────────────┐
│ source       ┆ study_id   │
│ ---          ┆ ---        │
│ str          ┆ str        │
╞══════════════╪════════════╡
│ gwas_catalog ┆ GCST000001 │
│ gwas_catalog ┆ GCST000001 │
└──────────────┴────────────┘


In [3]:
raw = pl.concat(gwas_catalog.iter_sumstats(files.row(0, named=True), batch_size=1))
assert raw.height == 2 and raw["p"].to_list() == [0.0, 0.8]
assert raw["se"].null_count() == 2
assert raw["effect_allele"][0] == "G"  # A2 in this observed source schema
assert raw["z"][0] == -1.35001010447145
assert {f["name"]: f["value"] for f in raw["raw_fields"][0]}["P"] == "0"
print(
    raw.select(
        "source", "study_id", "source_row", "chrom", "pos", "effect_allele", "beta", "se", "z", "p"
    )
)
print("provenance columns:", [c for c in raw.columns if c.startswith("source_")])

shape: (2, 10)
┌──────────────┬────────────┬────────────┬───────┬───┬──────────┬──────┬──────────┬─────┐
│ source       ┆ study_id   ┆ source_row ┆ chrom ┆ … ┆ beta     ┆ se   ┆ z        ┆ p   │
│ ---          ┆ ---        ┆ ---        ┆ ---   ┆   ┆ ---      ┆ ---  ┆ ---      ┆ --- │
│ str          ┆ str        ┆ i64        ┆ str   ┆   ┆ f64      ┆ f64  ┆ f64      ┆ f64 │
╞══════════════╪════════════╪════════════╪═══════╪═══╪══════════╪══════╪══════════╪═════╡
│ gwas_catalog ┆ GCST000001 ┆ 1          ┆ X     ┆ … ┆ -3.07292 ┆ null ┆ -1.35001 ┆ 0.0 │
│ gwas_catalog ┆ GCST000001 ┆ 2          ┆ X     ┆ … ┆ -3.07292 ┆ null ┆ -1.35001 ┆ 0.8 │
└──────────────┴────────────┴────────────┴───────┴───┴──────────┴──────┴──────────┴─────┘
provenance columns: ['source_file_id', 'source_url', 'source_sha256', 'source_row', 'source_etag']


The reader preserves source chromosome spelling and numeric precision, zero P,
missing measurements and original text in `raw_fields`. It does not reconstruct
P or Z, flip alleles, or infer a reference allele. `p_max` is inclusive and
explicit; log-P is preserved separately so extreme significance survives P
underflow. These raw region strings use inclusive source positions, unlike
BED's zero-based half-open intervals.


In [4]:
significant = pl.concat(gwas_catalog.iter_sumstats(source, p_max=5e-8))
regional = pl.concat(gwas_catalog.iter_sumstats(source, scope="regions", regions=["chrX:100-100"]))
assert significant.height == 1 and regional.height == 2
assert regional["chrom"].to_list() == ["X", "X"]
try:
    list(gwas_catalog.iter_sumstats(source, regions=["X:1-200"]))
except ValueError as exc:
    print("Explicit region scope required:", str(exc))
logp = work / "logp.tsv"
logp.write_text(
    "hm_chrom\thm_pos\thm_other_allele\thm_effect_allele\tneg_log_10_p_value\tbeta\n2\t10\tA\tT\t400\tNA\n"
)
print(
    pl.concat(gwas_catalog.iter_sumstats(logp, p_max=5e-8)).select(
        "p", "neg_log_10_p_value", "beta"
    )
)

Explicit region scope required: regions require scope='regions'
shape: (1, 3)
┌──────┬────────────────────┬──────┐
│ p    ┆ neg_log_10_p_value ┆ beta │
│ ---  ┆ ---                ┆ ---  │
│ f64  ┆ f64                ┆ f64  │
╞══════╪════════════════════╪══════╡
│ null ┆ 400.0              ┆ null │
└──────┴────────────────────┴──────┘


In [5]:
# A successful file can contain zero retained rows. It still gets a receipt.
# A missing file is a failure, not an empty success. Repair it and resume.
missing = directory / "later.h.tsv.gz"
resume_manifest = work / "resume_manifest.txt"
resume_manifest.write_text(alias.as_uri() + "\n" + missing.as_uri() + "\n")
resume_files = gwas_catalog.list_datasets(resume_manifest)
run = work / "catalog_run"
try:
    gwas_catalog.ingest_sumstats(resume_files, run, p_max=5e-8, workers=1)
except gwas_catalog.IngestError as exc:
    partial = exc.result
    print(
        "First run:", partial.succeeded, "succeeded,", partial.failed, "failed; rows", partial.rows
    )
    assert (partial.succeeded, partial.failed, partial.rows) == (1, 1, 0)
receipts = list((run / "receipts").glob("*.json"))
success = next(p for p in receipts if json.loads(p.read_text())["status"] == "ok")
before = success.read_bytes(), success.stat().st_mtime_ns
missing.write_bytes(gzip.compress((header + row).encode()))
result = gwas_catalog.ingest_sumstats(resume_files, run, p_max=5e-8, workers=2, batch_size=1)
assert before == (success.read_bytes(), success.stat().st_mtime_ns)
assert (result.succeeded, result.failed, result.rows) == (2, 0, 1)
print("Resumed:", result.succeeded, "succeeded; rows", result.rows)
print(
    "Receipt:",
    {k: json.loads(success.read_text())[k] for k in ["status", "rows", "scope", "p_max"]},
)
loaded = gwas_catalog.load_ingest_result(run)
assert loaded.scan().collect().height == 1
# Stray files cannot enter the verified scan; only committed receipts count.
(run / "artifacts" / "uncommitted.parquet").write_bytes(b"not a valid artifact")
assert loaded.scan().collect().height == 1
try:
    gwas_catalog.ingest_sumstats(resume_files, run, p_max=0.1)
except ValueError:
    print("Changed filters require a new run directory.")

First run: 1 succeeded, 1 failed; rows 0
Resumed: 2 succeeded; rows 1
Receipt: {'status': 'ok', 'rows': 0, 'scope': 'genomewide', 'p_max': 5e-08}
Changed filters require a new run directory.


In [6]:
# Harvest's identity and local_root resolve are source-specific.
stem = "21833088-GCST001198-EFO_0003885_buildGRCh37"
url = f"https://fixture.invalid/dl/{stem}/{stem}.tsv.gz"
harvest_manifest = work / "harvest_manifest.txt"
harvest_manifest.write_text(url + "\n")
harvest_files = gwas_harvest.list_datasets(harvest_manifest)
raw_directory = work / "harvest_raw" / stem
raw_directory.mkdir(parents=True)
(raw_directory / f"{stem}.tsv.gz").write_bytes(gzip.compress((header + row).encode()))
harvest_result = gwas_harvest.ingest_sumstats(
    harvest_files,
    work / "harvest_run",
    local_root=work / "harvest_raw",
    scope="regions",
    regions={stem: ["X:1-200"]},
)
assert harvest_files["gcst_id"][0] == "GCST001198"
assert harvest_files["study_id"][0] == stem
print(harvest_files.select("source", "study_id", "gcst_id"))
print(harvest_result.scan().collect().select("study_id", "p", "z", "se"))

shape: (1, 3)
┌──────────────┬─────────────────────────────────┬────────────┐
│ source       ┆ study_id                        ┆ gcst_id    │
│ ---          ┆ ---                             ┆ ---        │
│ str          ┆ str                             ┆ str        │
╞══════════════╪═════════════════════════════════╪════════════╡
│ gwas_harvest ┆ 21833088-GCST001198-EFO_000388… ┆ GCST001198 │
└──────────────┴─────────────────────────────────┴────────────┘
shape: (1, 4)
┌─────────────────────────────────┬─────┬──────────┬──────┐
│ study_id                        ┆ p   ┆ z        ┆ se   │
│ ---                             ┆ --- ┆ ---      ┆ ---  │
│ str                             ┆ f64 ┆ f64      ┆ f64  │
╞═════════════════════════════════╪═════╪══════════╪══════╡
│ 21833088-GCST001198-EFO_000388… ┆ 0.0 ┆ -1.35001 ┆ null │
└─────────────────────────────────┴─────┴──────────┴──────┘


In [7]:
# A truncated gzip never becomes a successful zero-row or partial artifact.
broken = work / "broken.tsv.gz"
broken.write_bytes(source.read_bytes()[:-6])
try:
    # Consume to EOF: partially yielded batches are invalid after any error.
    list(gwas_catalog.iter_sumstats(broken, batch_size=1))
except (EOFError, ValueError):
    print("Truncated gzip rejected before completion.")

Truncated gzip rejected before completion.


## Live production recipes — not executed

Save the exact `list_datasets` table before an acquisition; aliases are separate
file identities and must not be silently guessed away. `required_study_ids`
makes missing requested studies visible as failures. Keep receipts, `run.json`,
input identities and artifacts together. `allow_partial=True` returns a partial
result intentionally; always inspect `failed` before treating it as complete.

```python
files = gwas_catalog.list_datasets(study_ids=["GCST90000001"])
files.write_parquet("catalog_files.parquet")
result = gwas_catalog.ingest_sumstats(
    files, "catalog_all", required_study_ids=["GCST90000001"],
    workers=4, batch_size=100_000)  # no default P filter

regional = gwas_catalog.ingest_sumstats(
    files, "catalog_regions", scope="regions",
    regions={"GCST90000001": ["1:100000-200000"]},
    backend="auto", p_max=5e-8)
# auto may use indexed BGZF/tabix ranges when usable; full is an explicit
# whole-file fallback, tabix requires a usable index. Remote indexed receipts
# record region_sha256/region_row rather than claiming a full source digest.

harvest_files = gwas_harvest.list_datasets("harvest_manifest.txt")
result = gwas_harvest.ingest_sumstats(harvest_files, "harvest_all", local_root="raw")
```

`backend="tabix"` and byte-range server behavior are live/native integration
paths, not claimed to be validated by the local fixtures above. Full downloads
can be very large. `panukbb.sumstats_region(s)` is a separate indexed cohort
API; see [source catalog](17_source_catalog.ipynb).
